In [0]:
-- Gold OEWS measurement fact view
-- Grain: one row per OEWS series, year, and period.

CREATE OR REPLACE VIEW workforce_analytics.gold.fact_oews_measurement
COMMENT 'Analytics-ready long-form OEWS observations enriched with occupation, geography, industry, and measurement names.'
AS

WITH occupation_lookup AS (
    SELECT
        occupation_code,
        MAX(soc_code) AS soc_code,
        MAX(occupation_name) AS occupation_name
    FROM workforce_analytics.silver.bls_oews_occupations
    GROUP BY occupation_code
),

area_lookup AS (
    SELECT
        area_code,
        MAX(area_name) AS area_name,
        COUNT(*) AS area_lookup_record_count
    FROM workforce_analytics.silver.bls_oews_areas
    GROUP BY area_code
),

industry_lookup AS (
    SELECT
        industry_code,
        MAX(industry_name) AS industry_name
    FROM workforce_analytics.silver.bls_oews_industries
    GROUP BY industry_code
),

datatype_lookup AS (
    SELECT
        datatype_code,
        MAX(datatype_name) AS datatype_name
    FROM workforce_analytics.silver.bls_oews_datatypes
    GROUP BY datatype_code
)

SELECT
    observation.series_id,
    observation.year,
    observation.period,

    CASE
        WHEN observation.period IN ('A01', 'M13')
            THEN 'annual'
        WHEN observation.period RLIKE '^M(0[1-9]|1[0-2])$'
            THEN 'monthly'
        WHEN observation.period RLIKE '^S0[1-2]$'
            THEN 'semiannual'
        ELSE 'other'
    END AS period_type,

    CASE
        WHEN observation.period IN ('A01', 'M13')
            THEN MAKE_DATE(observation.year, 1, 1)
        WHEN observation.period RLIKE '^M(0[1-9]|1[0-2])$'
            THEN MAKE_DATE(
                observation.year,
                CAST(SUBSTRING(observation.period, 2, 2) AS INT),
                1
            )
        WHEN observation.period = 'S01'
            THEN MAKE_DATE(observation.year, 1, 1)
        WHEN observation.period = 'S02'
            THEN MAKE_DATE(observation.year, 7, 1)
        ELSE NULL
    END AS observation_date,

    series.seasonal,
    series.series_title,

    series.area_code,
    area.area_name,
    area.area_lookup_record_count,

    series.industry_code,
    industry.industry_name,

    series.occupation_code,
    COALESCE(
        series.soc_code,
        occupation.soc_code
    ) AS soc_code,
    occupation.occupation_name,

    series.datatype_code,
    datatype.datatype_name,

    observation.value,
    observation.value_raw,
    observation.footnote_codes,

    observation.source_file_path,
    observation.run_id,
    observation.processed_at

FROM workforce_analytics.silver.bls_oews_observations AS observation

INNER JOIN workforce_analytics.silver.bls_oews_series AS series
    ON observation.series_id = series.series_id

LEFT JOIN occupation_lookup AS occupation
    ON series.occupation_code = occupation.occupation_code

LEFT JOIN area_lookup AS area
    ON series.area_code = area.area_code

LEFT JOIN industry_lookup AS industry
    ON series.industry_code = industry.industry_code

LEFT JOIN datatype_lookup AS datatype
    ON series.datatype_code = datatype.datatype_code

WHERE observation.value IS NOT NULL;
